<a href="https://colab.research.google.com/github/isabelhendrick-sudo/Agriculture_data_visualisation/blob/main/weather_visualisation_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from matplotlib.colors import to_rgb


BG = "#F4F2ED"
EMPTY = "#E4E1DA"
INK = "#555555"
SOFT = "#8C8C8C"
SERIF = "DejaVu Serif"


TEMP_EDGES = [0, 5, 10, 15, 20, 25, 30]
TEMP_LABELS = ["below 0°C", "0–5°C", "5–10°C", "10–15°C",
               "15–20°C", "20–25°C", "25–30°C", "above 30°C"]
TEMP_COLOURS = ["#2B2F7F", "#1F6FB2", "#5EA8D9", "#AED6EE",
                "#F8D39A", "#F6A94F", "#EE7322", "#D7262B"]


CONDITIONS = ["Sunny", "Partly cloudy", "Cloudy", "Rainy", "Heavy rain", "Snowy"]
CONDITION_GREYS = ["#CFCFCF", "#9E9E9E", "#BDBDBD", "#6E6E6E", "#4D4D4D", "#2E2E2E"]
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def classify(row):
    """Turn rainfall / snow / cloud cover into a single weather condition."""
    if row["snow"] > 0:
        return "Snowy"
    if row["rainfall"] >= 20:
        return "Heavy rain"
    if row["rainfall"] >= 1:
        return "Rainy"
    if row["cloud"] >= 7:
        return "Cloudy"
    if row["cloud"] >= 3:
        return "Partly cloudy"
    return "Sunny"


def load_data(csv_path):
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    df["date"] = pd.to_datetime(df[["year", "month", "day"]])
    df = df.sort_values("date").reset_index(drop=True)
    df["condition"] = df.apply(classify, axis=1)
    df["band"] = np.digitize(df["tavg"], TEMP_EDGES)

    first_monday = df["date"].min() - pd.Timedelta(days=df["date"].min().weekday())
    df["col"] = (df["date"] - first_monday).dt.days // 7
    df["row"] = df["date"].dt.weekday
    return df


CSV_PATH = r"C:\Users\korea\Documents\Homework\weather_project\weather dataset.csv"
df = load_data(CSV_PATH)
df.head()


def _line(ax, xs, ys, c, lw, z):
    ax.plot(xs, ys, color=c, lw=lw, solid_capstyle="round", zorder=z)


def icon_sun(ax, x, y, s, c, lw=0.9, z=5):
    ax.add_patch(Circle((x, y), 0.12 * s, color=c, zorder=z, lw=0))
    for a in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        _line(ax, [x + 0.18 * s * np.cos(a), x + 0.27 * s * np.cos(a)],
              [y + 0.18 * s * np.sin(a), y + 0.27 * s * np.sin(a)], c, lw, z)


def icon_cloud(ax, x, y, s, c, z=5):
    for cx, cy, r in [(-0.13, 0.02, 0.12), (0.02, -0.06, 0.16), (0.16, 0.03, 0.11)]:
        ax.add_patch(Circle((x + cx * s, y + cy * s), r * s, color=c, zorder=z, lw=0))
    ax.add_patch(Rectangle((x - 0.13 * s, y + 0.02 * s), 0.29 * s, 0.12 * s,
                           color=c, zorder=z, lw=0))


def icon_partly(ax, x, y, s, c, bg, lw=0.9, z=5):
    icon_sun(ax, x - 0.1 * s, y - 0.1 * s, 0.8 * s, c, lw, z)
    icon_cloud(ax, x + 0.06 * s, y + 0.08 * s, 0.86 * s, bg, z + 1)
    icon_cloud(ax, x + 0.06 * s, y + 0.08 * s, 0.72 * s, c, z + 2)


def icon_rain(ax, x, y, s, c, heavy=False, lw=0.9, z=5):
    icon_cloud(ax, x, y - 0.09 * s, 0.85 * s, c, z)
    offsets = [-0.15, -0.05, 0.05, 0.15] if heavy else [-0.11, 0.0, 0.11]
    length = 0.16 if heavy else 0.11
    for dx in offsets:
        _line(ax, [x + dx * s, x + (dx - 0.04) * s],
              [y + 0.12 * s, y + (0.12 + length) * s], c, lw * (1.3 if heavy else 1), z)


def icon_snow(ax, x, y, s, c, lw=0.9, z=5):
    for a in (0, np.pi / 3, 2 * np.pi / 3):
        dx, dy = 0.26 * s * np.cos(a), 0.26 * s * np.sin(a)
        _line(ax, [x - dx, x + dx], [y - dy, y + dy], c, lw * 1.2, z)
    ax.add_patch(Circle((x, y), 0.05 * s, color=c, zorder=z, lw=0))


def draw_icon(ax, condition, x, y, s, c, bg, lw=0.9):
    if condition == "Sunny":
        icon_sun(ax, x, y, s, c, lw)
    elif condition == "Partly cloudy":
        icon_partly(ax, x, y, s, c, bg, lw)
    elif condition == "Cloudy":
        icon_cloud(ax, x, y + 0.03 * s, s, c)
    elif condition == "Rainy":
        icon_rain(ax, x, y, s, c, False, lw)
    elif condition == "Heavy rain":
        icon_rain(ax, x, y, s, c, True, lw)
    elif condition == "Snowy":
        icon_snow(ax, x, y, s, c, lw)


def ink_for(colour):
    """White icons on dark cells, grey icons on light cells."""
    r, g, b = to_rgb(colour)
    return "white" if 0.299 * r + 0.587 * g + 0.114 * b < 0.78 else "#5F6B73"


def clean(ax):
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])


In [ ]:
def draw_grid(ax, df, weeks):
    n = len(weeks)
    ax.set_xlim(0, n)
    ax.set_ylim(8.7, -2.3)
    clean(ax)
    gap = 0.06

    # Empty background cells
    for col in range(n):
        for row in range(7):
            ax.add_patch(Rectangle((col + gap, row + gap), 1 - 2 * gap, 1 - 2 * gap,
                                   color=EMPTY, lw=0, zorder=1))
    # Day cells
    for _, d in df.iterrows():
        colour = TEMP_COLOURS[d["band"]]
        ax.add_patch(Rectangle((d["col"] + gap, d["row"] + gap), 1 - 2 * gap, 1 - 2 * gap,
                               color=colour, lw=0, zorder=2))
        draw_icon(ax, d["condition"], d["col"] + 0.5, d["row"] + 0.5, 0.95,
                  ink_for(colour), colour, lw=0.7)

    # Weekday labels
    for row, name in enumerate(WEEKDAYS):
        ax.text(-0.3, row + 0.5, name, ha="right", va="center",
                fontsize=9, color=INK, family=SERIF)

    # Weekly average temperature and week number above each column
    for col, w in weeks.iterrows():
        colour = TEMP_COLOURS[int(np.digitize(w["avg"], TEMP_EDGES))]
        ax.text(col + 0.5, -1.75, f"{w['avg']:.0f}°", ha="center", va="center",
                fontsize=7, color=INK, fontweight="bold")
        ax.add_patch(Rectangle((col + gap, -1.35), 1 - 2 * gap, 0.18,
                               color=colour, lw=0, zorder=2))
        ax.text(col + 0.5, -0.55, f"{col + 1}", ha="center", va="center",
                fontsize=6.5, color=SOFT, family=SERIF)
    ax.text(-0.3, -1.75, "Week avg", ha="right", va="center", fontsize=8, color=SOFT)
    ax.text(-0.3, -0.55, "Week", ha="right", va="center", fontsize=8, color=SOFT)

    # Month spans underneath
    for month, m in df.groupby(df["date"].dt.month):
        c0, c1 = m["col"].min(), m["col"].max() + 1
        ax.plot([c0 + 0.25, c1 - 0.25], [7.35, 7.35], color=SOFT, lw=0.8)
        ax.text((c0 + c1) / 2, 7.95, m["date"].iloc[0].strftime("%b"),
                ha="center", va="center", fontsize=9.5, color=INK, family=SERIF)

def draw_week_curve(ax, weeks):
    n = len(weeks)
    x = np.arange(n) + 0.5
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.fill_between(x, weeks["min"], weeks["avg"], color="#5EA8D9", alpha=0.85, lw=0)
    ax.fill_between(x, weeks["avg"], weeks["max"], color="#F6A94F", alpha=0.85, lw=0)
    ax.plot(x, weeks["avg"], color="white", lw=1.2)
    ax.set_xlim(0, n)
    ax.set_xticks([])
    ax.yaxis.tick_right()
    ax.tick_params(axis="y", colors=SOFT, labelsize=8, length=0)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}°C")
    ax.grid(True, axis="y", color="#D8D5CE", lw=0.8)
    ax.set_axisbelow(True)
    first = weeks.iloc[0]
    for label, value in [("Max of the week", first["max"]),
                         ("Average of the week", first["avg"]),
                         ("Min of the week", first["min"])]:
        ax.annotate(label, (0.5, value), xytext=(-12, 0), textcoords="offset points",
                    ha="right", va="center", fontsize=8.5, color=INK, family=SERIF,
                    annotation_clip=False)

def draw_week_panel(ax, heading, colour, week_no, w, df_week):
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    clean(ax)
    ax.add_patch(Rectangle((0, 0), 1, 1, color="#EAE7E0", lw=0))
    ax.add_patch(Rectangle((0, 0), 1, 0.16, color=colour, lw=0))
    ax.text(0.5, 0.08, heading, ha="center", va="center", color="white",
            fontsize=11, family=SERIF, fontweight="bold")
    start, end = df_week["date"].min(), df_week["date"].max()
    ax.text(0.5, 0.25, f"Week {week_no}", ha="center", va="center",
            fontsize=13, color=INK, family=SERIF)
    ax.text(0.5, 0.33, f"{start:%d %b} – {end:%d %b}", ha="center", va="center",
            fontsize=9, color=SOFT)
    top = df_week["condition"].mode().iloc[0]
    ax.set_aspect("auto")
    # icon drawn in a square inset so circles stay round
    ins = ax.inset_axes([0.25, 0.29, 0.5, 0.33])
    ins.set_xlim(0, 1); ins.set_ylim(1, 0); ins.set_aspect("equal"); clean(ins)
    ins.patch.set_alpha(0)
    draw_icon(ins, top, 0.5, 0.52, 1.25, colour, "#EAE7E0", lw=2.2)
    ax.text(0.5, 0.79, f"{w['avg']:.1f}°C", ha="center", va="center",
            fontsize=22, color=INK, family=SERIF)
    ax.text(0.5, 0.9, f"average temperature\nmostly {top.lower()}", ha="center",
            va="center", fontsize=8, color=SOFT, linespacing=1.4)

def draw_share_bar(ax, title, labels, counts, colours, icons=False):
    total = counts.sum()
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    clean(ax)
    ax.text(0, 0.05, title, ha="left", va="center", fontsize=10.5,
            color=INK, family=SERIF)
    ax.plot([0, 1], [0.2, 0.2], color="#C9C6BF", lw=0.8)
    left = 0.0
    for label, count, colour in zip(labels, counts, colours):
        if count == 0:
            continue
        width = count / total
        ax.add_patch(Rectangle((left, 0.45), width, 0.18, color=colour, lw=0))
        mid = left + width / 2
        ax.text(mid, 0.36, f"{100 * width:.0f}%", ha="center", va="center",
                fontsize=7.5, color=SOFT)
        if icons:
            ins = ax.inset_axes([mid - 0.03, 0.66, 0.06, 0.34], transform=ax.transData)
            ins.set_xlim(0, 1); ins.set_ylim(1, 0); ins.set_aspect("equal"); clean(ins)
            draw_icon(ins, label, 0.5, 0.5, 1.0, "#7A7A7A", BG, lw=1.0)
        left += width




In [ ]:

def draw_legend(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    clean(ax)
    ax.text(0, 0.02, "Legend", fontsize=13, color=INK, family=SERIF, va="center")
    ax.plot([0, 0.9], [0.05, 0.05], color="#C9C6BF", lw=0.8)
    ax.text(0, 0.09, "Daily average\ntemperature", fontsize=8, color=SOFT, va="center")
    y = 0.15
    for label, colour in zip(reversed(TEMP_LABELS), reversed(TEMP_COLOURS)):
        ax.add_patch(Rectangle((0, y - 0.012), 0.12, 0.024, color=colour, lw=0))
        ax.text(0.2, y, label, fontsize=8, color=INK, va="center")
        y += 0.037
    y += 0.03
    ax.plot([0, 0.9], [y, y], color="#C9C6BF", lw=0.8)
    ax.text(0, y + 0.04, "Weather\nconditions", fontsize=8, color=SOFT, va="center")
    y += 0.1
    for cond in CONDITIONS:
        ins = ax.inset_axes([0, y - 0.02, 0.14, 0.04], transform=ax.transData)
        ins.set_xlim(0, 1); ins.set_ylim(1, 0); ins.set_aspect("equal"); clean(ins)
        draw_icon(ins, cond, 0.5, 0.5, 1.0, "#7A7A7A", BG, lw=0.8)
        ax.text(0.2, y, cond, fontsize=8, color=INK, va="center")
        y += 0.045
    ax.text(0, y + 0.03, "Conditions derived\nfrom rainfall, snow\nand cloud cover",
            fontsize=7, color=SOFT, va="top", linespacing=1.4)

def make_infographic(df, title, subtitle):
    weeks = df.groupby("col").agg(avg=("tavg", "mean"), max=("tmax", "max"),
                                  min=("tmin", "min"), days=("tavg", "size"))
    weeks = weeks.reindex(range(weeks.index.max() + 1))
    n = len(weeks)

    fig_w, fig_h = 22, 12.5
    fig = plt.figure(figsize=(fig_w, fig_h), facecolor=BG)

    left, width = 0.165, 0.665
    grid_h = width * fig_w * 11 / n / fig_h
    grid_b = 0.25
    ax_grid = fig.add_axes([left, grid_b, width, grid_h])
    ax_curve = fig.add_axes([left, grid_b + grid_h + 0.005, width, 0.18])
    panel_l = left + width + 0.03
    ax_legend = fig.add_axes([0.012, grid_b, 0.075, 0.6])

    draw_grid(ax_grid, df, weeks)
    draw_week_curve(ax_curve, weeks)
    draw_legend(ax_legend)

    # Coldest / warmest among weeks with at least 5 days of data
    full = weeks[weeks["days"] >= 5]
    panel_h = (grid_h + 0.185) / 2 - 0.01
    for i, (heading, colour, week) in enumerate([
            ("Coldest week", "#5EA8D9", full["avg"].idxmin()),
            ("Warmest week", "#F6A94F", full["avg"].idxmax())]):
        ax_p = fig.add_axes([panel_l, grid_b + grid_h + 0.185 - (i + 1) * (panel_h + 0.02) + 0.01,
                             0.985 - panel_l, panel_h])
        draw_week_panel(ax_p, heading, colour, week + 1, weeks.loc[week],
                        df[df["col"] == week])

    # Share bars along the bottom
    cond_counts = df["condition"].value_counts().reindex(CONDITIONS, fill_value=0)
    band_counts = df["band"].value_counts().reindex(range(8), fill_value=0)
    days = len(df)
    ax_c = fig.add_axes([left, 0.06, width * 0.47, 0.13])
    ax_t = fig.add_axes([left + width * 0.53, 0.06, width * 0.47, 0.13])
    draw_share_bar(ax_c, f"Weather conditions, share of {days} days",
                   CONDITIONS, cond_counts.to_numpy(), CONDITION_GREYS, icons=True)
    draw_share_bar(ax_t, f"Temperature bands, share of {days} days",
                   TEMP_LABELS, band_counts.to_numpy(), TEMP_COLOURS)

    fig.text(left, 0.965, title, fontsize=24, color=INK, family=SERIF, va="center")
    fig.text(left, 0.93, subtitle, fontsize=11, color=SOFT, va="center")
    return fig

fig = make_infographic(df, "A year of weather, 2024", "Station 146")
fig.savefig("weather_infographic.png", dpi=150, facecolor=BG)
print("Saved weather_infographic.png next to this notebook")


